In [ ]:
from ase.io import read, write
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
plt.rcParams.update({
    "font.family": 'sans-serif', 
    "font.size": 14,         
    "axes.titlesize": 14,        
    "axes.titleweight": "bold",   
    "axes.labelweight": "normal",  
    "lines.linewidth": 3,        
    "legend.fontsize": 14       
})

In [ ]:
am26=read("../../data/am26.extxyz", index=":")

In [ ]:
def extract_true_labels(testset_name):
    true_energies = []
    true_forces = []
    for i in range(len(testset_name)):
        true_energies.append(testset_name[i].info['energy_per_at'])
        true_forces.append(testset_name[i].get_forces())

    true_energies = np.array(true_energies)
    true_forces = np.concatenate(true_forces).flatten()
    return true_energies, true_forces

In [ ]:
# calculate RelRMSEs
# here shown for carbon, repeat for other systems as needed

# input model names, and corresponding energy and force RMSEs
model_names=["..."] 
e_rmse=[]
f_rmse=[]

# extract true energies and forces
carbon=[structure for structure in am26 if structure.info["system"]=="carbon"]
true_energies, true_forces=extract_true_labels(carbon)

std_energy = np.std(true_energies)*1000
std_forces = np.std(true_forces)*1000
normalised_e_rmse=[x / std_energy *100 for x in e_rmse]
normalised_f_rmse = [x / std_forces *100 for x in f_rmse]
carbon_e_relrmse=np.array(normalised_e_rmse)
carbon_f_relrmse=np.array(normalised_f_rmse)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap


def plot_heatmap_with_dot(model_names, column_labels, error_grid, mean_errors, std_errors,
                          heatmap_label="Energy RelRMSE (%)",
                          dot_label="Mean Energy RelRMSE (%)",
                          cmap_colors=None,
                          heatmap_vmin=None,
                          heatmap_vmax=None,
                          dot_color="#404040",
                          row_height_scale=1.0,
                          cbar_height=0.005,
                          cbar_gap=0.01,
                          drop_label_offset=0.001,
                          name="energy.pdf"):
    n_models, n_columns = error_grid.shape

    if cmap_colors is None:
        cmap_colors = ["#ecf4fb", "#c6dbef", "#74a9cf", "#2b8cbe", "#045a8d"]
    cmap = LinearSegmentedColormap.from_list("custom_cmap", cmap_colors)

    fig_h = n_models * 0.5 * row_height_scale + 1.2
    fig = plt.figure(figsize=(8, fig_h))

    left_margin   = 0.22
    right_margin = 0.05
    top_margin    = 0.06
    bottom_margin = 0.10

    total_w = 1.0 - left_margin - right_margin
    total_h = 1.0 - top_margin - bottom_margin

    hm_frac   = 0.55
    gap_frac  = 0.015
    dot_frac  = 0.4

    hm_w   = total_w * hm_frac
    dot_w  = total_w * dot_frac
    hm_x0  = left_margin
    dot_x0 = hm_x0 + hm_w + total_w * gap_frac

    cbar_h      = cbar_height
    cbar_gap    = cbar_gap
    cbar_w_frac = 0.75
    cbar_w      = hm_w * cbar_w_frac
    cbar_x0     = hm_x0 + (hm_w - cbar_w) / 2

    plot_h  = total_h - cbar_gap - cbar_h - 0.06
    plot_y0 = bottom_margin + cbar_gap + cbar_h + 0.06
    cbar_y0 = plot_y0 - cbar_gap - cbar_h

    ax_heat = fig.add_axes([hm_x0,  plot_y0, hm_w,  plot_h])
    ax_dot  = fig.add_axes([dot_x0, plot_y0, dot_w, plot_h])
    ax_cbar = fig.add_axes([cbar_x0, cbar_y0, cbar_w, cbar_h])

    im = ax_heat.imshow(error_grid, cmap=cmap, aspect="auto", vmin=heatmap_vmin, vmax=heatmap_vmax)

    ylim = (-0.5, n_models - 0.5)

    ax_heat.set_yticks(np.arange(n_models))
    ax_heat.set_yticklabels(model_names)
    ax_heat.set_ylim(ylim)
    ax_heat.invert_yaxis()

    ax_heat.set_xticks(np.arange(n_columns))
    ax_heat.set_xticklabels(column_labels)
    ax_heat.xaxis.tick_top()
    ax_heat.tick_params(axis='x', length=0, pad=7)

    for label in ax_heat.get_xticklabels():
        if '_' in label.get_text() or any(c in label.get_text() for c in ['₂','₁','₀']):
            label.set_y(label.get_position()[1] - drop_label_offset)

    threshold = (heatmap_vmin + heatmap_vmax) / 2

    for i in range(n_models):
        for j in range(n_columns):
            value = error_grid[i, j]

            text_color = "white" if value > threshold else "black"

            ax_heat.text(
                j, i,
                f"{value:.1f}",
                ha="center",
                va="center",
                color=text_color
            )

    for x in np.arange(0.5, n_columns - 1):
        ax_heat.axvline(x, color="white", linewidth=1.05, zorder=3)
    for y in np.arange(0.5, n_models - 1):
        ax_heat.axhline(y, color="white", linewidth=1.05, zorder=3)

    for spine in ax_heat.spines.values():
        spine.set_linewidth(1)
        spine.set_zorder(4)

    cbar = fig.colorbar(im, cax=ax_cbar, orientation="horizontal")
    cbar.set_label(heatmap_label, labelpad=5)

    if heatmap_vmax == 150:
        cbar.set_ticks([0, 50, 100, 150])
    elif heatmap_vmax == 60:
        cbar.set_ticks([0, 20, 40, 60])

    y = np.arange(n_models)
    ax_dot.errorbar(mean_errors, y, xerr=std_errors,
                fmt='o', color=dot_color,
                markersize=6.5, capsize=2,
                elinewidth=1.5, capthick=1.5,
                zorder=3)

    ax_dot.set_ylim(ylim)
    ax_dot.invert_yaxis()
    ax_dot.set_yticks([])
    ax_dot.set_yticklabels([])

    if heatmap_vmax == 100:
        ax_dot.set_xticks([0, 25, 50, 75, 100])
    elif heatmap_vmax == 60:
        ax_dot.set_xticks([0, 20, 40, 60])

    ax_dot.set_xlabel(dot_label, labelpad=12)
    ax_dot.set_xlim(0, heatmap_vmax * 1.1)

    for spine in ax_dot.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)

    ax_dot.set_facecolor("white")
    ax_dot.tick_params(axis='x', width=1, length=4, direction='out')

    for x_tick in ax_dot.get_xticks()[1:-1]:
        ax_dot.axvline(x_tick, color='#e0e0e0', linewidth=0.4, linestyle='-', zorder=0)

    for y_boundary in np.arange(0.5, n_models - 1):
        ax_dot.axhline(y_boundary, color='#ebebeb', linewidth=0.4, linestyle='-', zorder=0)

    ax_dot.set_axisbelow(True)
    
    plt.savefig(name, dpi=300, format='pdf', bbox_inches='tight')
    plt.show()

In [ ]:
column_labels = [r"$a$-C", r"$a$-Si", r"$a$-SiO$_2$", r"$a$-GST", r"$a$-LiPS"]
energy_data = np.array([[carbon_e_relrmse[i], silicon_e_relrmse[i], silica_e_relrmse[i], gst_e_relrmse[i], lips_e_relrmse[i]] for i in range(len(carbon_e_relrmse))])
length = len(carbon_e_relrmse)

mean_e_relrmse=[]
std_e_relrmse=[]
for i in range(length):
    mean_e_relrmse.append(np.mean([carbon_e_relrmse[i], silicon_e_relrmse[i], silica_e_relrmse[i], gst_e_relrmse[i], lips_e_relrmse[i]] ))
    std_e_relrmse.append(np.std([carbon_e_relrmse[i], silicon_e_relrmse[i], silica_e_relrmse[i], gst_e_relrmse[i], lips_e_relrmse[i]]))
mean_e_relrmse = np.array(mean_e_relrmse)
std_e_relrmse  = np.array(std_e_relrmse)

# Rank by performance
order = np.argsort(mean_e_relrmse)

energy_data = energy_data[order]
mean_e_relrmse = mean_e_relrmse[order]
std_e_relrmse  = std_e_relrmse[order]
model_ordered = np.array(model_names)[order]
print("Model ranking (best to worst):", model_ordered)

# Plotting the heatmap with dot plot
plot_heatmap_with_dot(
    model_names=model_ordered,
    column_labels=column_labels,
    error_grid=energy_data,
    mean_errors=mean_e_relrmse,
    std_errors=std_e_relrmse,
    heatmap_vmin=0, heatmap_vmax=160)